In [ ]:
# read fasta
from Bio import SeqIO
from plants_sm.io.pickle import read_pickle

def read_fasta(file_path):
    records = []
    for record in SeqIO.parse(file_path, "fasta"):
        records.append(record)
    return records

def write_fasta(records, output_file_path):
    """
    Write a list of SeqRecord objects to a FASTA file.

    Args:
        records (list): List of SeqRecord objects to write.
        output_file_path (str): Path to the output FASTA file.
    """
    with open(output_file_path, "w") as output_handle:
        SeqIO.write(records, output_handle, "fasta")

def separate_partitions_in_fasta(records, split_file, output_sufix):
    split_04 = read_pickle(split_file)
    train_ids = list(split_04[0][0])
    validation_ids = list(split_04[0][1])
    test_ids = list(split_04[0][2])

    train_records = []
    validation_records = []
    test_records = []

    for record in records:
        if record.id in train_ids:
            train_records.append(record)
        elif record.id in validation_ids:
            validation_records.append(record)
        elif record.id in test_ids:
            test_records.append(record)

    write_fasta(train_records, f"train_{output_sufix}.fasta")
    write_fasta(validation_records, f"validation_{output_sufix}.fasta")
    write_fasta(test_records, f"test_{output_sufix}.fasta")


records = read_fasta("../unique_enzymes_curated.fasta")
separate_partitions_in_fasta(records, "../../splits/splits_0_4_proteins_train_val_test.pkl", "40")
separate_partitions_in_fasta(records, "../../splits/splits_0_8_proteins_train_val_test.pkl", "80")
separate_partitions_in_fasta(records, "../../splits/splits_0_6_proteins_train_val_test.pkl", "60")
separate_partitions_in_fasta(records, "../../splits/splits_0_2_proteins_train_val_test.pkl", "20")

In [1]:
import os
import pandas as pd
import glob
import re

# Get all CSV files in the directory
all_files = glob.glob('*chunk_*.csv')

# Extract base names (e.g., 'train_vs_test_40' from 'train_vs_test_40_chunk_27.csv')
base_names = set()
for file in all_files:
    # Use regex to extract the base name
    match = re.match(r'(.+)_chunk_\d+\.csv', file)
    if match:
        base_names.add(match.group(1))

# Concatenate files for each base name
for base in base_names:
    files = [f for f in all_files if f.startswith(base)]
    if not files:
        print(f"No files found for {base}")
        continue

    # Read and concatenate all chunks
    dfs = [pd.read_csv(file) for file in files]
    combined = pd.concat(dfs, ignore_index=True)

    # Save the combined file
    output_file = f'{base}_combined.csv'
    combined.to_csv(os.path.join('combined_statistics', output_file), index=False)
    print(f"Combined {len(files)} chunks into {output_file}")


Combined 20 chunks into val_vs_test_20_combined.csv
Combined 20 chunks into train_vs_val_20_combined.csv
Combined 20 chunks into train_vs_test_20_combined.csv
